## Comparing dataset projections

In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [ ]:
# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Elevation (reprojection standard)
#### Copernicus DEM (PRISM)

In [ ]:
# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

# Create a single DEM image and clip it
elevation = (
    dataset
    .select('DEM')
    .first()
    .clip(panama_geom)
)

elevation_vis = {
    'min': 0,
    'max': 2500,
    'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff'],
}

Map.addLayer(elevation, elevation_vis, 'Panama DEM')

### Dynamic World LULC

In [20]:
START = "2024-01-01"
END = "2024-12-31"

# 10m resolution
dw_col = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(panama_geom)
    .filterDate(START, END)
)

VIS_PALETTE = [
    "419bdf", "397d49", "88b053", "7a87c6",
    "e49635", "dfc35a", "c4281b", "a59b8f", "b39fe1"
]

dw_reprojected = dw_col.first().select('trees').reproject(
    crs = elevation.projection(), scale = dw_col.first().projection().nominalScale()
)

Map.addLayer(
    dw_reprojected.clip(panama_geom), 
    {"min": 0, "max": 8, "palette": VIS_PALETTE},
    "Dynamic World Reprojected"
) 

In [ ]:
dw_col.first().bandNames().getInfo()

['water',
 'trees',
 'grass',
 'flooded_vegetation',
 'crops',
 'shrub_and_scrub',
 'built',
 'bare',
 'snow_and_ice',
 'label']

In [ ]:
dw_col.first().select("label").getInfo()

{'type': 'Image',
 'bands': [{'id': 'label',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 255},
   'dimensions': [4208, 10942],
   'crs': 'EPSG:32617',
   'crs_transform': [10, 0, 267490, 0, 10, 790400]}],
 'version': 1781622907260212,
 'id': 'GOOGLE/DYNAMICWORLD/V1/20240102T155529_20240102T155524_T17NKJ',
 'properties': {'system:time_start': 1704211278983,
  'dynamicworld_algorithm_version': '3.5',
  'qa_algorithm_version': '1',
  'system:footprint': {'type': 'LinearRing',
   'coordinates': [[-82.72841426480845, 8.136771857649837],
    [-82.72842570784877, 8.136772476104516],
    [-83.11010012039296, 8.134968787589656],
    [-83.11014186544926, 8.134932054540728],
    [-83.11018786683341, 8.134900694730696],
    [-83.1101905679813, 8.1348859136969],
    [-83.10768426408046, 7.640376010751643],
    [-83.10533961787318, 7.145861650623313],
    [-83.10530261758133, 7.1458203514389975],
    [-83.1052710199704, 7.145774805088946],
    [-83.10525612

### Distance to protected areas

In [37]:
pa_fc = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(panama_geom)

pa_raster = ee.Image().byte().paint(pa_fc, 1)

dist_pa = (
    pa_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .clip(panama_geom)
)

# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_dist_pa = (
    dist_pa.resample("bilinear")
    .reproject(collection_projection)
    .rename("dist_pa_reprojected")
    .clip(panama_geom)
)

Map.addLayer(pa_fc, {}, "Protected Areas")
Map.addLayer(reprojected_dist_pa, {}, "Distance Protected Areas")

In [ ]:
projection_info = pa_fc.first().geometry().projection()
crs_code = projection_info.crs().getInfo()
full_metadata = projection_info.getInfo()
print(f"CRS Code: {crs_code}")
print("Full Metadata:", full_metadata)

CRS Code: EPSG:4326
Full Metadata: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}


In [ ]:
projection = pa_fc.first().geometry().projection()
nominal_scale = projection.nominalScale().getInfo()
print(f"Nominal Scale: {nominal_scale} meters")

Nominal Scale: 111319.49079327357 meters


In [39]:
# pa_fc.projection().getInfo()
reprojected_dist_pa.projection().nominalScale().getInfo()

30.922080775909325

### Distance to roads

In [40]:
roads = ee.FeatureCollection("projects/deforestation-495419/assets/Panama_OSM_Roads")

roads_raster = ee.Image().byte().paint(roads, 1)

distance_roads = (
    roads_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .rename("dist_roads_m")
    .clip(panama_geom)
)

# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_distance_roads = (
    distance_roads.resample("bilinear")
    .reproject(collection_projection)
    .rename("dist_roads_reprojected")
    .clip(panama_geom)
)

Map.addLayer(roads_raster, {"palette": ["black"]}, "Roads")
Map.addLayer(reprojected_distance_roads, {"min": 0, "max": 5000}, "Distance to Roads")

In [ ]:
projection_info = roads.first().geometry().projection()
crs_code = projection_info.crs().getInfo()
full_metadata = projection_info.getInfo()
print(f"CRS Code: {crs_code}")
print("Full Metadata:", full_metadata)

CRS Code: EPSG:4326
Full Metadata: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}


In [ ]:
projection = roads.first().geometry().projection()
nominal_scale = projection.nominalScale().getInfo()
print(f"Nominal Scale: {nominal_scale} meters")

Nominal Scale: 111319.49079327357 meters


In [42]:
reprojected_distance_roads.projection().nominalScale().getInfo()

30.922080775909325

### Distance to urban and rural areas

In [25]:
# 1000m resolution
smod = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030").select("smod_code")

urban = smod.gte(21).And(smod.lte(30))
rural = smod.gte(11).And(smod.lte(13))

urban_img = urban.unmask(0).toByte()
rural_img = rural.unmask(0).toByte()

distance_urban = urban_img.distance(ee.Kernel.euclidean(50000, "meters")).clip(panama_geom)
distance_rural = rural_img.distance(ee.Kernel.euclidean(50000, "meters")).clip(panama_geom)

# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_urban = (
    distance_urban.resample("bilinear")
    .reproject(collection_projection)
    .rename("urban_reprojected")
    .clip(panama_geom)
)

reprojected_rural = (
    distance_rural.resample("bilinear")
    .reproject(collection_projection)
    .rename("rural_reprojected")
    .clip(panama_geom)
)

# Map.addLayer(distance_urban, {"min": 0, "max": 25000}, "Distance Urban")
# Map.addLayer(distance_rural, {"min": 0, "max": 25000}, "Distance Rural")
Map.addLayer(reprojected_urban, {"min": 0, "max": 25000}, "Urban Distance Reprojected")
Map.addLayer(reprojected_rural, {"min": 0, "max": 25000}, "Rural Distance Reprojected")

# Map.addLayer(urban.selfMask(), {"palette": ["red"]}, "Urban")
# Map.addLayer(rural.selfMask(), {"palette": ["blue"]}, "Rural")

In [ ]:
smod.projection().getInfo()

{'type': 'Projection',
 'wkt': 'PROJCS["World_Mollweide", \n  GEOGCS["WGS 84", \n    DATUM["WGS_1984", \n      SPHEROID["WGS 84", 6378137.0, 298.257223563, AUTHORITY["EPSG","7030"]], \n      AUTHORITY["EPSG","6326"]], \n    PRIMEM["Greenwich", 0.0], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH]], \n  PROJECTION["Mollweide"], \n  PARAMETER["semi_minor", 6378137.0], \n  PARAMETER["false_easting", 0.0], \n  PARAMETER["false_northing", 0.0], \n  PARAMETER["central_meridian", 0.0], \n  UNIT["m", 1.0], \n  AXIS["Easting", EAST], \n  AXIS["Northing", NORTH]]',
 'transform': [1000, 0, -18041000, 0, -1000, 9000000]}

In [29]:
# reprojected_rural.projection().nominalScale().getInfo()
reprojected_urban.projection().nominalScale().getInfo()


30.922080775909325

### Forest type (dry vs wet)

In [32]:
# 10m resolution
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()

landcover = worldcover.select("Map")

forest = landcover.eq(10)

ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")

dry_ecoregion = ecoregions.filter(
    ee.Filter.eq("ECO_NAME", "Panamanian dry forests")
)

dry_mask = ee.Image.constant(0).paint(dry_ecoregion, 1).selfMask()

dry_forest = forest.updateMask(dry_mask)
wet_forest = forest.updateMask(dry_mask.unmask(0).Not())

# 111,319m resolution
classified = ee.Image(0)
classified = classified.where(wet_forest, 1)
classified = classified.where(dry_forest, 2)

# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_classified = (
    classified.resample("bilinear")
    .reproject(collection_projection)
    .rename("classified_reprojected")
    .clip(panama_geom)
)

Map.addLayer(
    reprojected_classified.clip(panama_geom),
    {"min": 0, "max": 2, "palette": ["white", "006400", "d4a017"]},
    "Forest Type"
)

In [ ]:
projection_info = dry_ecoregion.first().geometry().projection()
crs_code = projection_info.crs().getInfo()
full_metadata = projection_info.getInfo()
print(f"CRS Code: {crs_code}")
print("Full Metadata:", full_metadata)

CRS Code: EPSG:4326
Full Metadata: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}


In [ ]:
projection = dry_ecoregion.first().geometry().projection()
nominal_scale = projection.nominalScale().getInfo()
print(f"Nominal Scale: {nominal_scale} meters")

Nominal Scale: 111319.49079327357 meters


### Generate Map

In [ ]:
Map

Map(bottom=15906.0, center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position',…